### Imports

In [1]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv
from IPython.display import display, Markdown
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from operator import itemgetter
from deepeval import evaluate
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase
from deepeval.models import DeepEvalBaseLLM


C:\Users\shiva\AppData\Local\Temp\ipykernel_34904\3311367263.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


In [2]:
import sys
import types

# --- QUICK FIX FOR RAGAS BUG ---
# Ragas is crashing because it's looking for an old VertexAI module.
# We're just giving it a fake empty module so it shuts up and loads!
dummy_chat = types.ModuleType("langchain_community.chat_models.vertexai")
dummy_chat.ChatVertexAI = type("ChatVertexAI", (object,), {})
sys.modules["langchain_community.chat_models.vertexai"] = dummy_chat

import langchain_community.llms
langchain_community.llms.VertexAI = type("VertexAI", (object,), {})

In [3]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from datasets import Dataset
from ragas.run_config import RunConfig

C:\Users\shiva\AppData\Local\Temp\ipykernel_34904\2323399901.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy
C:\Users\shiva\AppData\Local\Temp\ipykernel_34904\2323399901.py:2: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy


In [4]:
import asyncio
import json
from typing import List, Dict, Any
import re

### Parsing and Chunking

In [ ]:
data_dir = Path("..") / "data"
chunk_size = 1000
chunk_overlap = 0

# --- Fix hyphenation artifact from PDF parsing ---
def fix_hyphenation(text):
    return re.sub(r'(\w+)-\n(\w+)', r'\1\2', text)

# --- Chunking function ---
def create_simple_chunks(file_path, chunk_size=3500, chunk_overlap=200):
    print("Loading pages:", file_path)
    loader = PyMuPDFLoader(str(file_path))
    doc_pages = loader.load()

    # Fix hyphenation on each page
    for page in doc_pages:
        page.page_content = fix_hyphenation(page.page_content)

    print("Chunking pages:", file_path)
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    doc_chunks = splitter.split_documents(doc_pages)
    print("Finished processing:", file_path)
    print()
    return doc_chunks

# --- Load and chunk each paper separately ---
attention_chunks = create_simple_chunks(data_dir / "1706.03762v7.pdf", chunk_size=chunk_size, chunk_overlap=chunk_overlap)
rag_chunks       = create_simple_chunks(data_dir / "2005.11401v4.pdf", chunk_size=chunk_size, chunk_overlap=chunk_overlap)
gpt3_chunks      = create_simple_chunks(data_dir / "2005.14165v4.pdf", chunk_size=chunk_size, chunk_overlap=chunk_overlap)

# --- Combine all chunks ---
all_chunks = []
all_chunks.extend(attention_chunks)
all_chunks.extend(rag_chunks)
all_chunks.extend(gpt3_chunks)

print(f"Attention chunks : {len(attention_chunks)}")
print(f"RAG chunks       : {len(rag_chunks)}")
print(f"GPT-3 chunks     : {len(gpt3_chunks)}")
print(f"Total chunks     : {len(all_chunks)}")

Loading pages: ..\data\1706.03762v7.pdf
Chunking pages: ..\data\1706.03762v7.pdf
Finished processing: ..\data\1706.03762v7.pdf

Loading pages: ..\data\2005.11401v4.pdf
Chunking pages: ..\data\2005.11401v4.pdf
Finished processing: ..\data\2005.11401v4.pdf

Loading pages: ..\data\2005.14165v4.pdf
Chunking pages: ..\data\2005.14165v4.pdf
Finished processing: ..\data\2005.14165v4.pdf

Attention chunks : 48
RAG chunks       : 83
GPT-3 chunks     : 284
Total chunks     : 415


### Indexing

### If creating db for the first time

In [ ]:
embed_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

chroma_db = Chroma.from_documents(
    documents=all_chunks,
    collection_name="rag_papers",
    embedding=embed_model,
    collection_metadata={"hnsw:space": "cosine"},
    persist_directory="./my_db"
)

print("Vector DB created!")

### If db is already created, we can load it here

In [ ]:
# embed_model = HuggingFaceEmbeddings(
#     model_name="sentence-transformers/all-MiniLM-L6-v2"
# )

# chroma_db = Chroma(
#     persist_directory="./my_db",
#     collection_name="rag_papers",
#     embedding_function=embed_model
# )

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### Checking number of entries in ChromaDB

In [8]:
# --- Check total entries in ChromaDB ---
collection = chroma_db._collection
print(f"Total entries in ChromaDB: {collection.count()}")

# --- Get all documents ---
all_entries = collection.get()
print(f"Total IDs: {len(all_entries['ids'])}")

# --- Check for duplicates ---
contents = all_entries['documents']
seen = {}
duplicates = []

for i, content in enumerate(contents):
    fingerprint = content[:200]
    if fingerprint in seen:
        duplicates.append({
            "original_idx": seen[fingerprint],
            "duplicate_idx": i,
            "content_preview": content[:100]
        })
    else:
        seen[fingerprint] = i

print(f"\nDuplicates found: {len(duplicates)}")
for d in duplicates:
    print(f"\nOriginal  idx: {d['original_idx']}")
    print(f"Duplicate idx: {d['duplicate_idx']}")
    print(f"Preview: {d['content_preview']}")

# --- Check metadata ---
print("\n--- Sample metadata ---")
for i, meta in enumerate(all_entries['metadatas'][:5]):
    print(f"Entry {i}: {meta}")

Total entries in ChromaDB: 415
Total IDs: 415

Duplicates found: 1

Original  idx: 46
Duplicate idx: 47
Preview: The
Law
will
never
be
perfect
,
but
its
application
should
be
just
-
this
is
what
we
are
missing
,
i

--- Sample metadata ---
Entry 0: {'total_pages': 15, 'producer': 'pdfTeX-1.40.25', 'keywords': '', 'modDate': 'D:20240410211143Z', 'author': '', 'creationDate': 'D:20240410211143Z', 'title': '', 'creator': 'LaTeX with hyperref', 'file_path': '..\\data\\1706.03762v7.pdf', 'source': '..\\data\\1706.03762v7.pdf', 'moddate': '2024-04-10T21:11:43+00:00', 'trapped': '', 'format': 'PDF 1.5', 'subject': '', 'creationdate': '2024-04-10T21:11:43+00:00', 'page': 0}
Entry 1: {'author': '', 'trapped': '', 'keywords': '', 'page': 0, 'format': 'PDF 1.5', 'creator': 'LaTeX with hyperref', 'subject': '', 'producer': 'pdfTeX-1.40.25', 'creationDate': 'D:20240410211143Z', 'title': '', 'creationdate': '2024-04-10T21:11:43+00:00', 'total_pages': 15, 'file_path': '..\\data\\1706.03762v7.pdf', 'mo

### Retrieval

Here, we are setting up the retriever. Using MMR to have diversity in retrieval.

In [9]:
similarity_retriever = chroma_db.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 10,  "fetch_k": 30}
)

### RAG Chain Creation

In [10]:
rag_prompt = """You are an assistant who is an expert in question-answering tasks.
Answer the following question using only the following pieces of retrieved context.
If the answer is not in the context, do not make up answers, just say that you don't know.
Keep the answer detailed and well formatted based on the information from the context.

Question: {question}

Context: {context}

Answer:"""
rag_prompt_template = ChatPromptTemplate.from_template(rag_prompt)

#### RAG Chain Creation with sources and citations

In [27]:
load_dotenv(Path("..") / ".env")
llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Chain 1: generates the answer string
src_rag_response_chain = (
    {
        "context": itemgetter("context") | RunnableLambda(format_docs),
        "question": itemgetter("question")
    }
    | rag_prompt_template
    | llm
    | StrOutputParser()
)

# Chain 2: retrieves docs + generates answer, returns BOTH
rag_chain_w_sources = (
    {
        "context": similarity_retriever,
        "question": RunnablePassthrough()
    }
    | RunnablePassthrough.assign(response=src_rag_response_chain)
)

### Testing the Whole RAG Pipeline on the provided questions

#### Q1

In [28]:
query = "What are the main components of a RAG model, and how do they interact?"
result = rag_chain_w_sources.invoke(query)
# Answer
display(Markdown(result["response"]))

# Sources
print("\n--- SOURCES ---")
for doc in result["context"]:
    print(f"File : {doc.metadata.get('source', 'unknown')}")
    print(f"Page : {doc.metadata.get('page', 'unknown')}")
    print(f"Text : {doc.page_content}")
    print()

**Main Components of a RAG Model and Their Interaction**

A RAG (Retrieval-Augmented Generation) model consists of two primary components: a retriever and a generator.

### 1. Retriever

The retriever is a component that returns a set of relevant text documents given a query. It is denoted as `pη(z|x)`, where `η` are the parameters of the retriever, `z` represents the retrieved text documents, and `x` is the input query. The retriever uses a query encoder to encode the input query and a document encoder to encode the text documents. The retriever then computes the similarity between the query and each document to retrieve the top-K most relevant documents.

### 2. Generator

The generator is a component that generates the target sequence `y` given the input query `x`, the retrieved text documents `z`, and the previous generated tokens `y1:i−1`. It is denoted as `pθ(yi|x, z, y1:i−1)`, where `θ` are the parameters of the generator. The generator uses the retrieved text documents as additional context to generate the target sequence.

**Interaction between the Retriever and Generator**

The retriever and generator interact in the following way:

1. The input query `x` is passed through the retriever to retrieve a set of relevant text documents `z`.
2. The retrieved text documents `z` are used as additional context by the generator to generate the target sequence `y`.
3. The generator uses the input query `x`, the retrieved text documents `z`, and the previous generated tokens `y1:i−1` to generate the next token `yi`.
4. The process is repeated until the target sequence `y` is generated.

**Key Features of the RAG Model**

The RAG model has several key features that enable it to generate high-quality text:

* **Retrieval-Augmentation**: The RAG model uses retrieval to augment the input query with relevant text documents, which provides additional context for the generator to generate the target sequence.
* **Non-Parametric Memory**: The RAG model uses a non-parametric memory to store the retrieved text documents, which allows it to update its knowledge as the world changes.
* **Trainable Parameters**: The RAG model has trainable parameters for the BERT-base query and document encoder of DPR, as well as the BART-large generator, which enables it to learn from data and improve its performance over time.


--- SOURCES ---
File : ..\data\2005.11401v4.pdf
Page : 1
Text : the non-parametric memory can be replaced to update the models’ knowledge as the world changes.1
2
Methods
We explore RAG models, which use the input sequence x to retrieve text documents z and use them
as additional context when generating the target sequence y. As shown in Figure 1, our models
leverage two components: (i) a retriever pη(z|x) with parameters η that returns (top-K truncated)
distributions over text passages given a query x and (ii) a generator pθ(yi|x, z, y1:i−1) parametrized
1Code to run experiments with RAG has been open-sourced as part of the HuggingFace Transformers Library [66] and can be found at https://github.com/huggingface/transformers/blob/master/
examples/rag/. An interactive demo of RAG models can be found at https://huggingface.co/rag/
2

File : ..\data\2005.11401v4.pdf
Page : 5
Text : Table 2 shows our results on FEVER. For 3-way classiﬁcation, RAG scores are within 4.3% of
state-of-the-art

#### Q2

In [29]:
query = "What are the two sub-layers in each encoder layer of the Transformer model?"
result = rag_chain_w_sources.invoke(query)
# Answer
display(Markdown(result["response"]))

# Sources
print("\n--- SOURCES ---")
for doc in result["context"]:
    print(f"File : {doc.metadata.get('source', 'unknown')}")
    print(f"Page : {doc.metadata.get('page', 'unknown')}")
    print(f"Text : {doc.page_content}")
    print()

**Two sub-layers in each encoder layer of the Transformer model**

According to the provided context, each encoder layer in the Transformer model consists of two sub-layers:

1. **Multi-head self-attention mechanism**: This is the first sub-layer in each encoder layer. It is a self-attention mechanism that allows the model to attend to different parts of the input sequence simultaneously and weigh their importance.
2. **Position-wise fully connected feed-forward network**: This is the second sub-layer in each encoder layer. It is a simple, position-wise fully connected feed-forward network that applies a linear transformation to each position in the input sequence.

Both sub-layers are followed by a residual connection and layer normalization to facilitate the flow of information through the model.


--- SOURCES ---
File : ..\data\1706.03762v7.pdf
Page : 2
Text : Figure 1: The Transformer - model architecture.
The Transformer follows this overall architecture using stacked self-attention and point-wise, fully
connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,
respectively.
3.1
Encoder and Decoder Stacks
Encoder:
The encoder is composed of a stack of N = 6 identical layers. Each layer has two
sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, positionwise fully connected feed-forward network. We employ a residual connection [11] around each of
the two sub-layers, followed by layer normalization [1]. That is, the output of each sub-layer is
LayerNorm(x + Sublayer(x)), where Sublayer(x) is the function implemented by the sub-layer
itself. To facilitate these residual connections, all sub-layers in the model, as well as the embedding
layers, produce outputs of dimension dmodel = 512.
Decoder:

Fil

#### Q3

In [30]:
query = "Explain how positional encoding is implemented in Transformers and why it is necessary."
result = rag_chain_w_sources.invoke(query)
# Answer
display(Markdown(result["response"]))


# Sources
print("\n--- SOURCES ---")
for doc in result["context"]:
    print(f"File : {doc.metadata.get('source', 'unknown')}")
    print(f"Page : {doc.metadata.get('page', 'unknown')}")
    print(f"Text : {doc.page_content}")
    print()

**Positional Encoding in Transformers**

**Why is Positional Encoding necessary?**

Positional encoding is necessary in Transformers to provide the model with information about the position of each token in the input sequence. This is crucial because self-attention mechanisms, which are the core of Transformers, do not have inherent knowledge of the order of the input tokens. Without positional encoding, the model would not be able to distinguish between tokens at different positions in the sequence.

**How is Positional Encoding implemented in Transformers?**

In this work, we use a sinusoidal positional encoding scheme, which is a fixed function of the position and dimension. The encoding is calculated as follows:

PE(pos, 2i) = sin(pos / 10000^(2i/dmodel))
PE(pos, 2i+1) = cos(pos / 10000^(2i/dmodel))

where pos is the position, i is the dimension, and dmodel is the model dimension.

Each dimension of the positional encoding corresponds to a sinusoid, and the wavelengths form a geometric progression from 2π to 10000 · 2π. This allows the model to easily learn to attend by relative positions, since for any fixed offset k, PE(pos+k) can be represented as a linear function of PE(pos).

**Why was the sinusoidal positional encoding scheme chosen?**

We chose this function because we hypothesized it would allow the model to easily learn to attend by relative positions. We also experimented with using learned positional embeddings, but found that the two versions produced nearly identical results. We chose the sinusoidal version because it may allow the model to extrapolate to sequence lengths longer than the ones encountered during training.

**Comparison with other positional encoding schemes**

We also experimented with using learned positional embeddings, but found that the two versions produced nearly identical results. This suggests that the sinusoidal positional encoding scheme is a good choice for Transformers, and that learned positional embeddings may not be necessary.


--- SOURCES ---
File : ..\data\1706.03762v7.pdf
Page : 2
Text : Figure 1: The Transformer - model architecture.
The Transformer follows this overall architecture using stacked self-attention and point-wise, fully
connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,
respectively.
3.1
Encoder and Decoder Stacks
Encoder:
The encoder is composed of a stack of N = 6 identical layers. Each layer has two
sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, positionwise fully connected feed-forward network. We employ a residual connection [11] around each of
the two sub-layers, followed by layer normalization [1]. That is, the output of each sub-layer is
LayerNorm(x + Sublayer(x)), where Sublayer(x) is the function implemented by the sub-layer
itself. To facilitate these residual connections, all sub-layers in the model, as well as the embedding
layers, produce outputs of dimension dmodel = 512.
Decoder:

Fil

#### Q4

In [31]:
query = "Describe the concept of multi-head attention in the Transformer architecture. Why is it beneficial?"
result = rag_chain_w_sources.invoke(query)
# Answer
display(Markdown(result["response"]))

# Sources
print("\n--- SOURCES ---")
for doc in result["context"]:
    print(f"File : {doc.metadata.get('source', 'unknown')}")
    print(f"Page : {doc.metadata.get('page', 'unknown')}")
    print(f"Text : {doc.page_content}")
    print()

**Concept of Multi-Head Attention in the Transformer Architecture**

Multi-head attention is a mechanism used in the Transformer architecture to jointly attend to information from different representation subspaces at different positions. This is in contrast to single-head attention, where averaging inhibits this ability.

The multi-head attention mechanism is defined as:

MultiHead(Q, K, V) = Concat(head1, ..., headh)WO

where headi = Attention(QWQi, KWKi, VWVi)

The projections are parameter matrices WQi ∈ Rdmodel×dk, WK i ∈ Rdmodel×dk, WV i ∈ Rdmodel×dv, and WO ∈ Rhdv×dmodel.

**Why is Multi-Head Attention Beneficial?**

Multi-head attention is beneficial for several reasons:

1.  **Ability to jointly attend to information from different representation subspaces**: Multi-head attention allows the model to attend to different aspects of the input at the same time, which is not possible with single-head attention.
2.  **Reduced dimensionality**: By reducing the dimensionality of each head, the total computational cost is similar to that of single-head attention with full dimensionality.
3.  **Improved ability to learn long-distance dependencies**: Multi-head attention has been shown to improve the ability to learn long-distance dependencies in the input, which is particularly useful for tasks such as machine translation.

**Applications of Multi-Head Attention in the Transformer Architecture**

Multi-head attention is used in three different ways in the Transformer architecture:

1.  **Encoder-decoder attention layers**: The queries come from the previous decoder layer, and the keys and values come from the encoder output.
2.  **Self-attention mechanism**: The self-attention mechanism is used in the encoder to attend to different positions in the input sequence.
3.  **Point-wise, fully connected layers**: The point-wise, fully connected layers are used in both the encoder and decoder to transform the input and output of the self-attention mechanism.

Overall, multi-head attention is a key component of the Transformer architecture that allows the model to jointly attend to information from different representation subspaces at different positions, which is particularly useful for tasks such as machine translation.


--- SOURCES ---
File : ..\data\1706.03762v7.pdf
Page : 4
Text : output values. These are concatenated and once again projected, resulting in the final values, as
depicted in Figure 2.
Multi-head attention allows the model to jointly attend to information from different representation
subspaces at different positions. With a single attention head, averaging inhibits this.
MultiHead(Q, K, V ) = Concat(head1, ..., headh)W O
where headi = Attention(QW Q
i , KW K
i , V W V
i )
Where the projections are parameter matrices W Q
i
∈Rdmodel×dk, W K
i
∈Rdmodel×dk, W V
i
∈Rdmodel×dv
and W O ∈Rhdv×dmodel.
In this work we employ h = 8 parallel attention layers, or heads. For each of these we use
dk = dv = dmodel/h = 64. Due to the reduced dimension of each head, the total computational cost
is similar to that of single-head attention with full dimensionality.
3.2.3
Applications of Attention in our Model
The Transformer uses multi-head attention in three different ways:
• In "encoder-decoder attenti

#### Q5

In [32]:
query = "What is few-shot learning, and how does GPT-3 implement it during inference?"
result = rag_chain_w_sources.invoke(query)
# Answer
display(Markdown(result["response"]))

# Sources
print("\n--- SOURCES ---")
for doc in result["context"]:
    print(f"File : {doc.metadata.get('source', 'unknown')}")
    print(f"Page : {doc.metadata.get('page', 'unknown')}")
    print(f"Text : {doc.page_content}")
    print()

**Few-Shot Learning and GPT-3 Implementation**

**What is Few-Shot Learning?**

Few-shot learning is a setting where a model is given a few demonstrations of a task at inference time as conditioning, but no weight updates are allowed. This means that the model is expected to learn and adapt to a new task based on a limited number of examples, without requiring extensive fine-tuning or retraining.

**How Does GPT-3 Implement Few-Shot Learning?**

GPT-3 implements few-shot learning by providing the model with a few examples of context and completion, and then asking it to generate the completion for a new, unseen example. The model is expected to learn from these few examples and generate accurate completions.

**Key Features of GPT-3's Few-Shot Learning Implementation**

* The model is given a few examples of context and completion, typically in the range of 10 to 100.
* The model is expected to generate the completion for a new, unseen example based on the few examples provided.
* No weight updates are allowed, meaning that the model's parameters are not updated during inference time.
* The model's performance improves strongly with model size, with larger models achieving better results in the few-shot setting.

**Advantages of Few-Shot Learning**

* Major reduction in the need for task-specific data.
* Reduced potential to learn an overly narrow distribution from a large but narrow fine-tuning dataset.
* Improved sample efficiency, as the model can learn from a few examples rather than requiring extensive fine-tuning.

**Disadvantages of Few-Shot Learning**

* Results from this method have so far been much worse than state-of-the-art fine-tuned models.
* A small amount of task-specific data is still required.

**Comparison to Other Settings**

* Few-shot learning is distinct from zero-shot and one-shot learning, which involve providing no examples or only one example, respectively.
* Few-shot learning is related to meta-learning, which involves learning to learn from a broad distribution of tasks.

**Performance of GPT-3 in Few-Shot Learning**

* GPT-3 achieves 86.4% accuracy in the few-shot setting, an increase of over 18% from the previous state-of-the-art.
* The largest model achieves 65% accuracy in the few-shot setting, and demonstrates significant gains over zero-shot behavior.


--- SOURCES ---
File : ..\data\2005.14165v4.pdf
Page : 5
Text : hundreds of thousands of labeled examples are used. The main advantage of ﬁne-tuning is strong performance
on many benchmarks. The main disadvantages are the need for a new large dataset for every task, the potential
for poor generalization out-of-distribution [MPL19], and the potential to exploit spurious features of the
training data [GSL+18, NK19], potentially resulting in an unfair comparison with human performance. In
this work we do not ﬁne-tune GPT-3 because our focus is on task-agnostic performance, but GPT-3 can be
ﬁne-tuned in principle and this is a promising direction for future work.
• Few-Shot (FS) is the term we will use in this work to refer to the setting where the model is given a few
demonstrations of the task at inference time as conditioning [RWC+19], but no weight updates are allowed.
As shown in Figure 2.1, for a typical dataset an example has a context and a desired completion (for example

File : 

### Metrics Calculation

Test Questions list

In [33]:
test_questions = [
    "What are the main components of a RAG model, and how do they interact?",
    "What are the two sub-layers in each encoder layer of the Transformer model?",
    "Explain how positional encoding is implemented in Transformers and why it is necessary.",
    "Describe the concept of multi-head attention in the Transformer architecture. Why is it beneficial?",
    "What is few-shot learning, and how does GPT-3 implement it during inference?"
]

Answer generation

In [34]:
answers = []
contexts = []

for q in test_questions:
    result = rag_chain_w_sources.invoke(q)
    answers.append(result["response"])
    contexts.append([doc.page_content for doc in result["context"]])

print("Done generating answers!")

Done generating answers!


Checking number of tokens in retrieved contexts

In [35]:
import tiktoken

# Tokenizer (cl100k is used by most modern LLMs, close enough for estimation)
enc = tiktoken.get_encoding("cl100k_base")

print("=== Context Size Analysis ===\n")
for i, (q, ctx) in enumerate(zip(test_questions, contexts)):
    print(f"Q{i+1}: {q[:60]}")
    total_chars  = sum(len(c) for c in ctx)
    total_tokens = sum(len(enc.encode(c)) for c in ctx)
    print(f"   Chunks        : {len(ctx)}")
    print(f"   Total chars   : {total_chars}")
    print(f"   Total tokens  : {total_tokens}")
    for j, c in enumerate(ctx):
        print(f"   Chunk {j+1}      : {len(c)} chars | {len(enc.encode(c))} tokens")
    print()

=== Context Size Analysis ===

Q1: What are the main components of a RAG model, and how do they
   Chunks        : 10
   Total chars   : 7442
   Total tokens  : 2186
   Chunk 1      : 778 chars | 194 tokens
   Chunk 2      : 293 chars | 71 tokens
   Chunk 3      : 439 chars | 100 tokens
   Chunk 4      : 676 chars | 159 tokens
   Chunk 5      : 929 chars | 228 tokens
   Chunk 6      : 927 chars | 198 tokens
   Chunk 7      : 478 chars | 111 tokens
   Chunk 8      : 976 chars | 239 tokens
   Chunk 9      : 990 chars | 438 tokens
   Chunk 10      : 956 chars | 448 tokens

Q2: What are the two sub-layers in each encoder layer of the Tra
   Chunks        : 10
   Total chars   : 6890
   Total tokens  : 2017
   Chunk 1      : 930 chars | 208 tokens
   Chunk 2      : 945 chars | 239 tokens
   Chunk 3      : 402 chars | 79 tokens
   Chunk 4      : 484 chars | 110 tokens
   Chunk 5      : 282 chars | 78 tokens
   Chunk 6      : 956 chars | 448 tokens
   Chunk 7      : 918 chars | 188 tokens
   

Running Deepeval using GPT OSS model

In [36]:
MODEL_NAME = "openai/gpt-oss-120b"
MAX_CONCURRENT_REQUESTS = 1
DELAY_BETWEEN_BATCHES = 8.0
MAX_CONTEXT_CHARS = 4000
DELAY_BETWEEN_QUESTIONS = 5.0

# ── Groq Judge ────────────────────────────────────────────────────────────────
class GroqJudge(DeepEvalBaseLLM):
    def __init__(self, model_name: str):
        self.model_name = model_name
        self.model = ChatGroq(model_name=model_name, temperature=0)

    def load_model(self):
        return self.model

    def generate(self, prompt: str) -> str:
        return self.model.invoke(prompt).content

    async def a_generate(self, prompt: str) -> str:
        response = await self.model.ainvoke(prompt)
        return response.content

    def get_model_name(self) -> str:
        return self.model_name

# ── Context trimmer ───────────────────────────────────────────────────────────
def clean_context(contexts: List[str], max_chars: int = MAX_CONTEXT_CHARS) -> List[str]:
    return [c[:max_chars] for c in contexts]

# ── Async evaluator ───────────────────────────────────────────────────────────
async def evaluate_single(
    semaphore: asyncio.Semaphore,
    question: str,
    answer: str,
    contexts: List[str],
    judge: GroqJudge
) -> Dict[str, Any]:
    async with semaphore:
        safe_ctx = clean_context(contexts)

        test_case = LLMTestCase(
            input=question,
            actual_output=answer,
            retrieval_context=safe_ctx
        )

        faith_metric = FaithfulnessMetric(threshold=0.5, model=judge, async_mode=True)
        rel_metric   = AnswerRelevancyMetric(threshold=0.5, model=judge, async_mode=True)

        for attempt in range(3):                        # retry up to 3 times
            try:
                await faith_metric.a_measure(test_case)
                await asyncio.sleep(DELAY_BETWEEN_BATCHES)
                await rel_metric.a_measure(test_case)

                await asyncio.sleep(DELAY_BETWEEN_QUESTIONS)  # cool down before next Q

                return {
                    "question"         : question,
                    "faithfulness"     : faith_metric.score,
                    "answer_relevancy" : rel_metric.score,
                    "error"            : None
                }

            except Exception as e:
                err_str = str(e)
                if "429" in err_str and attempt < 2:
                    wait = 15 * (attempt + 1)           # 15s, then 30s
                    print(f"   ⏳ Rate limited, retrying in {wait}s... (attempt {attempt+1}/3)")
                    await asyncio.sleep(wait)
                else:
                    return {
                        "question"         : question,
                        "faithfulness"     : None,
                        "answer_relevancy" : None,
                        "error"            : err_str
                    }

# ── Main ──────────────────────────────────────────────────────────────────────
async def run_evaluation():
    judge = GroqJudge(model_name=MODEL_NAME)
    sem   = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)

    tasks = [
        evaluate_single(sem, q, a, ctx, judge)
        for q, a, ctx in zip(test_questions, answers, contexts)
    ]

    print(f"Evaluating {len(tasks)} questions...\n")
    results = await asyncio.gather(*tasks)

    # ── Print results ─────────────────────────────────────────────────────────
    eval_results = []
    for res in results:
        if res["error"]:
            print(f"❌ {res['question'][:60]}\n   Error: {res['error']}\n")
        else:
            print(f"✅ {res['question'][:60]}")
            print(f"   Faithfulness    : {res['faithfulness']}")
            print(f"   Answer Relevancy: {res['answer_relevancy']}\n")
            eval_results.append(res)

    if eval_results:
        avg_faith = sum(r["faithfulness"] for r in eval_results) / len(eval_results)
        avg_rel   = sum(r["answer_relevancy"] for r in eval_results) / len(eval_results)
        print(f"=== AVERAGE ({len(eval_results)}/{len(tasks)} successful) ===")
        print(f"Faithfulness    : {avg_faith:.3f}")
        print(f"Answer Relevancy: {avg_rel:.3f}")

# ── Run in Jupyter ────────────────────────────────────────────────────────────
await run_evaluation()

Output()

Evaluating 5 questions...



Output()

Output()

Output()

...

Output()

Output()

Output()

Output()

✅ What are the main components of a RAG model, and how do they
   Faithfulness    : 1.0
   Answer Relevancy: 0.8421052631578947

✅ What are the two sub-layers in each encoder layer of the Tra
   Faithfulness    : 1.0
   Answer Relevancy: 0.8571428571428571

✅ Explain how positional encoding is implemented in Transforme
   Faithfulness    : 1.0
   Answer Relevancy: 1.0

✅ Describe the concept of multi-head attention in the Transfor
   Faithfulness    : 1.0
   Answer Relevancy: 0.9230769230769231

✅ What is few-shot learning, and how does GPT-3 implement it d
   Faithfulness    : 1.0
   Answer Relevancy: 1.0

=== AVERAGE (5/5 successful) ===
Faithfulness    : 1.000
Answer Relevancy: 0.924
